In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    build_feature_combos,
    detect_device,
    dropna_splits,
    load_split_bundle,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data


In [2]:
DATA_SET = 'rand_D'


df_train, df_val, df_test = load_split_bundle(CLEAN_DATA, DATA_SET)

OUTPUT_DIR = make_run_dir()

## GPU auto-detect


In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
BATCH_SIZE = CFG['BATCH']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

A100-80GB  |  VRAM: 85 GB  |  batch=65,536  |  dtype=torch.bfloat16


## Hyperparameters


In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'LR={INIT_LR:.4f} (scaled from {BASE_LR} at batch {BASE_BATCH})  '
      f'BATCH={BATCH_SIZE:,}  WARMUP={WARMUP_EPOCHS} epochs')


MAX_EPOCHS=100  PATIENCE=30  LR=0.0160 (scaled from 0.001 at batch 4096)  BATCH=65,536  WARMUP=5 epochs


## Feature definitions


In [5]:
FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'iv_lag', 'd_iv_lag','vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

feature_combos = build_feature_combos(FEATURES_3, EXTRA_FEATURES, max_extra=3)

n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')


Total feature combinations: 130
  Base (3F):  1
  3F + 1:     9
  3F + 2:     36
  3F + 3:     84


## Drop rows with NaN and pre-allocate on GPU


In [6]:
required_cols = list(set(ALL_FEATURES + [TARGET]))
df_train, df_val, df_test, nan_stats = dropna_splits(df_train, df_val, df_test, required_cols)
print(f'Rows before: {nan_stats["before"]:,}  after: {nan_stats["after"]:,}  dropped: {nan_stats["dropped"]:,}')

gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Rows before: 1,204,505  after: 1,185,312  dropped: 19,193
Data on GPU  |  VRAM used: 0.52 GB / 85 GB  |  Free: 84.6 GB
Train: 829,665  Val: 237,102  Test: 118,545  Features: 12


## Analytic benchmark


In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 8.2488  RMSE = 0.008342
Coefficients: a = -0.165659, b = -0.000404, c = 0.032764


## Train all feature combinations


In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

print(f'\n{"=" * 70}')
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(f'{"=" * 70}\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 130 MODELS
  GPU: A100-80GB  |  Batch: 65,536  |  AMP: True  |  Max epochs: 100

  [  1/130] 3F                             SSE=10.8473  Gain=-31.5%  ep=100  10.5s  elapsed=0.2min
  [ 25/130] 3F+iv_lag+rho                  SSE=9.7231  Gain=-17.9%  ep=100  4.1s  elapsed=1.8min
  [ 50/130] 3F+vix_lag+iv_lag+gamma        SSE=13.5483  Gain=-64.2%  ep=100  4.1s  elapsed=3.5min
  [ 75/130] 3F+iv_lag+d_iv_lag+vix_mom_lag SSE=11.5591  Gain=-40.1%  ep=100  4.1s  elapsed=5.2min
  [100/130] 3F+d_iv_lag+vix_mom_lag+rho    SSE=14.1978  Gain=-72.1%  ep=100  4.1s  elapsed=6.9min
  [125/130] 3F+vix_mom+theta+rho           SSE=13.5363  Gain=-64.1%  ep=100  4.0s  elapsed=8.6min
  [130/130] 3F+theta+vega+rho              SSE=12.7535  Gain=-54.6%  ep=100  4.2s  elapsed=9.0min

Done: 9.0 min for 130 models (avg 4.1s/model)


## Results summary


In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,8.248751,0.000070,0.008342,0.003511,0.000742,0.001876,0.125530,None,None,None
1,3F,10.847255,0.000092,0.009566,0.005637,0.000041,0.003937,-0.149943,5.0s,-31.50%,None
2,3F+vix_lag,15.574365,0.000131,0.011462,0.007613,-0.000249,0.005542,-0.651076,4.1s,-88.81%,-43.58%
3,3F+iv_lag,16.226068,0.000137,0.011699,0.008384,-0.000141,0.006335,-0.720164,4.1s,-96.71%,-4.18%
4,3F+d_iv_lag,15.994222,0.000135,0.011616,0.008033,-0.000572,0.005946,-0.695586,4.0s,-93.90%,1.43%
...,...,...,...,...,...,...,...,...,...,...,...
127,3F+gamma+theta+vega,17.014893,0.000144,0.011980,0.008152,-0.000588,0.006037,-0.803789,4.1s,-106.27%,-58.42%
128,3F+gamma+theta+rho,12.865135,0.000109,0.010418,0.006783,0.000133,0.004900,-0.363864,4.0s,-55.96%,24.39%
129,3F+gamma+vega+rho,15.464578,0.000130,0.011422,0.007618,-0.000345,0.005744,-0.639437,4.1s,-87.48%,-20.21%
130,3F+theta+vega+rho,12.753514,0.000108,0.010372,0.006680,0.000097,0.004834,-0.352031,4.2s,-54.61%,17.53%


## Top 10 by Gain vs Hull-White


In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,3F+iv_lag+d_iv_lag,5,8.0061,0.008218,2.94,4.1
1,3F+d_iv_lag+gamma,5,8.5294,0.008482,-3.40,4.1
2,3F+d_iv_lag+vix_mom+rho,6,9.1831,0.008801,-11.33,4.0
3,3F+iv_lag+rho,5,9.7231,0.009057,-17.87,4.1
4,3F+d_iv_lag+theta,5,9.8455,0.009113,-19.36,4.1
5,3F+d_iv_lag+vega,5,9.8617,0.009121,-19.55,4.1
6,3F+iv_lag+d_iv_lag+theta,6,9.8929,0.009135,-19.93,4.0
7,3F+d_iv_lag+vix_mom,5,9.9012,0.009139,-20.03,4.1
8,3F+vix_lag+iv_lag+d_iv_lag,6,10.2350,0.009292,-24.08,4.0
9,3F+iv_lag+vix_mom+gamma,6,10.3177,0.009329,-25.08,4.1


## Best per group (3F, +1, +2, +3)


In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f'{nf_label}: {best["combo_name"]}')
    print(f'    SSE={best["SSE"]:.4f}  RMSE={best["RMSE"]:.6f}  Gain={best["Gain_vs_HW_%"]:.2f}%\n')

3F (base): 3F
    SSE=10.8473  RMSE=0.009566  Gain=-31.50%

+1 (4F): 3F+vega
    SSE=13.9855  RMSE=0.010862  Gain=-69.55%

+2 (5F): 3F+iv_lag+d_iv_lag
    SSE=8.0061  RMSE=0.008218  Gain=2.94%

+3 (6F): 3F+d_iv_lag+vix_mom+rho
    SSE=9.1831  RMSE=0.008801  Gain=-11.33%



## Summary statistics


In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 8.8 min (0.15 hr)
Models trained: 130
Best overall: 3F+iv_lag+d_iv_lag (Gain=2.94%)


## Cleanup


In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.67 GB / 85 GB
Total training time: 8.8 min for 130 models
